In [491]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

## Getting and Joining Data

In [492]:
user_profile = pd.read_csv('/Users/christianmonsalve/Desktop/CODE/Projects/Movie_Recs/user_data/user_data.csv')

In [493]:
data = pd.read_csv('/Users/christianmonsalve/Desktop/CODE/Projects/Movie_Recs/data/processed/new_movie_features.csv')

In [494]:
data = user_profile.merge(data, how='left', on='tconst')

In [495]:
data.head()

,tconst,userRating,primaryTitle,year,runtimeMinutes,averageRating,numVotes,hasRating,Action,Adult,...,Sport,Talk-Show,Thriller,Unknown,War,Western,logNumVotes,releaseDecade,runtime_bin,bayesRating
0,tt0076759,5.0,Star Wars: Episode IV - A New Hope,1977,121,8.6,1552665.0,1,1,0,...,0,0,0,0,0,0,14.255484,1970,4,8.599922
1,tt0080684,6.0,Star Wars: Episode V - The Empire Strikes Back,1980,124,8.7,1487175.0,1,1,0,...,0,0,0,0,0,0,14.212390,1980,4,8.699916
2,tt0086190,5.5,Star Wars: Episode VI - Return of the Jedi,1983,131,8.3,1195744.0,1,1,0,...,0,0,0,0,0,0,13.994280,1980,4,8.299912
3,tt0119654,3.5,Men in Black,1997,98,7.3,656632.0,1,1,0,...,0,0,0,0,0,0,13.394881,1990,2,7.299915
4,tt0120338,8.0,Titanic,1997,194,7.9,1375336.0,1,0,0,...,0,0,0,0,0,0,14.134209,1990,5,7.899938


## Find User Vector

In [496]:
features = data.select_dtypes(include=['float64', 'int64'])

In [497]:
user_mean = features['userRating'].mean()
print("User rating mean:", round(user_mean, 2))

User rating mean: 6.85


In [498]:
userRatings = data['userRating']
weight = userRatings - user_mean

In [499]:
features = features.drop(columns=['userRating', 'numVotes'])

In [500]:
to_standardize = ['year', 'runtimeMinutes', 'logNumVotes', 'averageRating', 'releaseDecade', 'runtime_bin', 'bayesRating']

In [501]:
scaler = MinMaxScaler()

In [502]:
features[to_standardize] = scaler.fit_transform(features[to_standardize])

In [503]:
weighted_features = features.multiply(weight, axis=0)

In [504]:
weighted_features.head()

,year,runtimeMinutes,averageRating,hasRating,Action,Adult,Adventure,Animation,Biography,Comedy,...,Sport,Talk-Show,Thriller,Unknown,War,Western,logNumVotes,releaseDecade,runtime_bin,bayesRating
0,-0.000000,-0.939803,-1.596970,-1.854545,-1.854545,-0.0,-1.854545,-0.0,-0.0,-0.000000,...,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-1.749388,-0.000000,-1.390909,-1.596916
1,-0.053409,-0.450369,-0.759596,-0.854545,-0.854545,-0.0,-0.854545,-0.0,-0.0,-0.000000,...,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.803100,-0.170909,-0.640909,-0.759573
2,-0.169318,-0.777948,-1.053535,-1.354545,-1.354545,-0.0,-1.354545,-0.0,-0.0,-0.000000,...,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-1.249008,-0.270909,-1.015909,-1.053475
3,-1.397727,-1.178624,-1.677273,-3.354545,-3.354545,-0.0,-3.354545,-0.0,-0.0,-3.354545,...,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-2.929904,-1.341818,-0.838636,-1.676991
4,0.477273,1.145455,0.763636,1.145455,0.000000,0.0,0.000000,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,1.069224,0.458182,1.145455,0.763575


In [505]:
weighted_features.shape

(88, 37)

In [506]:
user_vector_numerator = weighted_features.sum()

In [507]:
denominator = weight.abs().sum()

In [508]:
denominator

np.float64(118.39999999999998)

In [509]:
user_vector = user_vector_numerator / denominator

In [510]:
print(round(user_vector, 5))

year              0.04130
runtimeMinutes    0.03518
averageRating    -0.01300
hasRating        -0.00000
Action            0.00215
Adult             0.00000
Adventure         0.06396
Animation         0.02250
Biography         0.00000
Comedy           -0.06081
Crime            -0.03225
Documentary       0.00000
Drama            -0.02449
Family            0.00000
Fantasy          -0.02933
Film-Noir         0.00000
Game-Show         0.00000
History           0.00000
Horror           -0.01321
Music             0.00000
Musical           0.00000
Mystery           0.00545
News              0.00000
Reality-TV        0.00000
Romance           0.02849
Sci-Fi            0.08070
Short             0.00000
Sport             0.00000
Talk-Show         0.00000
Thriller         -0.00353
Unknown           0.00000
War               0.00000
Western           0.00000
logNumVotes       0.00338
releaseDecade     0.05679
runtime_bin       0.04259
bayesRating      -0.01029
dtype: float64


## Recommendations - Based on Cosine Similarity

In [511]:
movies = pd.read_csv('/Users/christianmonsalve/Desktop/CODE/Projects/Movie_Recs/data/processed/new_movie_features.csv')

In [512]:
lowerVotesQuantile = movies['logNumVotes'].quantile(0.25)

In [513]:
movies = movies.drop(columns='numVotes')

In [514]:
movies = movies[~movies['tconst'].isin(user_profile['tconst'])].query(f'year > 2000 & logNumVotes > {lowerVotesQuantile} & runtimeMinutes > 60')

In [515]:
movies[to_standardize] = scaler.transform(movies[to_standardize])

In [516]:
movies.head()

,tconst,primaryTitle,year,runtimeMinutes,averageRating,hasRating,Action,Adult,Adventure,Animation,...,Sport,Talk-Show,Thriller,Unknown,War,Western,logNumVotes,releaseDecade,runtime_bin,bayesRating
24265,tt0035423,Kate & Leopold,0.500000,0.486486,0.250000,1,0,0,0,0,...,0,0,0,0,0,0,0.714923,0.6,0.50,0.249871
45309,tt0062336,The Tango of the Widower and Its Distorting Mi...,0.895833,0.162162,0.250000,1,0,0,0,0,...,0,0,0,0,0,0,0.238794,1.0,0.00,0.240485
50155,tt0069049,The Other Side of the Wind,0.854167,0.513514,0.333333,1,0,0,0,0,...,0,0,0,0,0,0,0.519888,0.8,0.75,0.332410
51225,tt0070596,Socialist Realism,0.958333,0.216216,0.388889,1,0,0,0,0,...,0,0,0,0,0,0,0.213886,1.0,0.00,0.348227
59102,tt0082328,Embodiment of Evil,0.645833,0.324324,0.111111,1,0,0,0,0,...,0,0,0,0,0,0,0.399054,0.6,0.25,0.113033


In [517]:
print("Shape:", movies.shape)

Shape: (168638, 39)


In [518]:
user_vector_2d = np.array(user_vector).reshape(1, -1)

In [519]:
user_vector_2d.shape

(1, 37)

In [520]:
movies['cosineSimilarity'] = cosine_similarity(np.array(movies.loc[:, 'year':]), user_vector_2d)

In [521]:
alpha = 0.50
beta = 0.10
gamma = 0.20
phi = 0.20

In [522]:
movies['final_score'] = alpha * movies['cosineSimilarity'] + \
              beta * movies['bayesRating'] + \
              gamma * movies['logNumVotes']**2 + \
              phi * movies['year']

In [523]:
movies.head()

,tconst,primaryTitle,year,runtimeMinutes,averageRating,hasRating,Action,Adult,Adventure,Animation,...,Thriller,Unknown,War,Western,logNumVotes,releaseDecade,runtime_bin,bayesRating,cosineSimilarity,final_score
24265,tt0035423,Kate & Leopold,0.500000,0.486486,0.250000,1,0,0,0,0,...,0,0,0,0,0.714923,0.6,0.50,0.249871,0.071896,0.263158
45309,tt0062336,The Tango of the Widower and Its Distorting Mi...,0.895833,0.162162,0.250000,1,0,0,0,0,...,0,0,0,0,0.238794,1.0,0.00,0.240485,0.214654,0.321946
50155,tt0069049,The Other Side of the Wind,0.854167,0.513514,0.333333,1,0,0,0,0,...,0,0,0,0,0.519888,0.8,0.75,0.332410,0.283808,0.400035
51225,tt0070596,Socialist Realism,0.958333,0.216216,0.388889,1,0,0,0,0,...,0,0,0,0,0.213886,1.0,0.00,0.348227,0.190875,0.331076
59102,tt0082328,Embodiment of Evil,0.645833,0.324324,0.111111,1,0,0,0,0,...,0,0,0,0,0.399054,0.6,0.25,0.113033,0.236847,0.290742


In [524]:
sorted_movies_by_similarity = movies.sort_values(by='final_score', ascending=False)

In [525]:
movie_names = pd.read_csv('/Users/christianmonsalve/Desktop/CODE/Projects/Movie_Recs/data/processed/new_movie_features.csv')

In [526]:
movie_names = movie_names[~movie_names['tconst'].isin(user_profile['tconst'])]

In [527]:
movie_names.head()

,tconst,primaryTitle,year,runtimeMinutes,averageRating,numVotes,hasRating,Action,Adult,Adventure,...,Sport,Talk-Show,Thriller,Unknown,War,Western,logNumVotes,releaseDecade,runtime_bin,bayesRating
0,tt0000009,Miss Jerry,1894,45,5.2,232.0,1,0,0,0,...,0,0,0,0,0,0,5.451038,1890,1,5.375019
1,tt0000147,The Corbett-Fitzsimmons Fight,1897,100,5.3,584.0,1,0,0,0,...,1,0,0,0,0,0,6.371612,1890,2,5.369961
2,tt0000335,Soldiers of the Cross,1900,40,5.4,67.0,1,0,0,0,...,0,0,0,0,0,0,4.219508,1900,1,5.736371
3,tt0000502,Bohemios,1905,100,3.1,26.0,1,0,0,0,...,0,0,0,1,0,0,3.295837,1900,2,5.130992
4,tt0000574,The Story of the Kelly Gang,1906,70,6.0,1047.0,1,1,0,1,...,0,0,0,0,0,0,6.954639,1900,1,6.008528


In [528]:
movie_names = movie_names[['tconst', 'primaryTitle']]

In [529]:
sorted_movies_by_similarity.head(1)

,tconst,primaryTitle,year,runtimeMinutes,averageRating,hasRating,Action,Adult,Adventure,Animation,...,Thriller,Unknown,War,Western,logNumVotes,releaseDecade,runtime_bin,bayesRating,cosineSimilarity,final_score
654254,tt5950044,Superman,1.0,0.560811,0.444444,1,1,0,1,0,...,0,0,0,0,0.828925,1.0,0.75,0.444342,0.628049,0.695882


In [530]:
print("Top 10 Recommendations")
index = sorted_movies_by_similarity.head(50).tconst

Top 10 Recommendations


In [531]:
recommendations = movie_names.set_index('tconst').loc[index].reset_index()
recommendations.index += 1

In [532]:
recommendations

,tconst,primaryTitle
1,tt5950044,Superman
2,tt12037194,Furiosa: A Mad Max Saga
3,tt1375666,Inception
4,tt0816692,Interstellar
5,tt1392190,Mad Max: Fury Road
6,tt3748528,Rogue One: A Star Wars Story
7,tt1462764,Indiana Jones and the Dial of Destiny
8,tt1446714,Prometheus
9,tt1877832,X-Men: Days of Future Past
10,tt2975590,Batman v Superman: Dawn of Justice
